# Mode A: Baseline Pure NSGA-II

**Self-contained experiment notebook using DRY architecture.**

## Architecture (DRY Principle)

| Location | What | Example |
|----------|------|---------|
| `schedule_engine/notebooks/` | Reusable functions | `load_data()`, `run_nsga2()`, `plot_convergence()` |
| `src/schedule_engine/config/models.py` | Global time config | `quantum_minutes`, `opening_time` |
| **This notebook** | Mode-specific config | `POP_SIZE`, `NGEN`, experiment execution |

## Mode A: Pure NSGA-II
- No repair heuristics
- No local search
- No RL guidance
- Baseline for comparison

## 1. Imports (from `schedule_engine/notebooks/`)

In [1]:
from __future__ import annotations
import random
import numpy as np
from pathlib import Path

# DRY IMPORTS FROM schedule_engine/notebooks/
from schedule_engine.notebooks.core import load_data, create_random_individual
from schedule_engine.notebooks.core import course_aware_crossover, smart_mutation
from schedule_engine.notebooks.core import create_evaluator, get_constraint_breakdown
from schedule_engine.notebooks.core import run_nsga2, EvolutionConfig, get_best_individual
from schedule_engine.notebooks.viz import plot_convergence, plot_constraint_breakdown, print_summary

print(" All imports from schedule_engine/notebooks/ successful!")

 All imports from schedule_engine/notebooks/ successful!


## 2. Mode A Configuration (Inline - Mode-Specific)

In [2]:

# MODE A CONFIGURATION - Modify these as needed

from datetime import datetime

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Evolution config (inline - mode-specific)
config = EvolutionConfig(
    pop_size=50,
    ngen=100,
    cxpb=0.9,
    mutpb=0.2,
    fitness_weights=(-1.0, -0.01),  # (hard, soft) - both minimized
    verbose=True,
    log_interval=20,
)

# Paths - Organized by mode with timestamp
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(f"../output/mode_a_baseline/{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f" Mode A Config: pop={config.pop_size}, ngen={config.ngen}, cxpb={config.cxpb}")
print(f" Output: {OUTPUT_DIR}")

 Mode A Config: pop=50, ngen=100, cxpb=0.9
 Output: ../output/mode_a_baseline/20260122_123618


## 3. Load Data (using `schedule_engine/notebooks/data_loader`)

In [3]:
# Load all data with single function call
data = load_data(
    data_dir=DATA_DIR,
    opening_time="10:00",
    closing_time="17:00",
    closed_days=["Saturday"],
)

print(f" {data.summary()}")

[!warn] groups enrolled but courses missing

CE604: BCE5A, BCE5B, BCE5C, BCE5D, BCE5E, BCE5F (ltp null)

ENCE 256: BCE4A, BCE4B, BCE4C, BCE4D, BCE4E, BCE4F (ltp null)

ENIE 254: BIE4A, BIE4B (ltp null)

ME706: BME7A, BME7B (ltp null)

16 course enrollments skipped

 Courses: 668, Groups: 74, Instructors: 181, Rooms: 67, Quanta: 42


## 4. Test Population & Evaluation

In [4]:
# Test individual creation
test_ind = create_random_individual(data)
print(f" Individual has {len(test_ind)} genes")

# Test evaluation
evaluate = create_evaluator(data)
test_fitness = evaluate(test_ind)
print(f" Test fitness: hard={test_fitness[0]}, soft={test_fitness[1]}")

 Individual has 713 genes
 Test fitness: hard=5364.0, soft=2870.0


## 5. Run NSGA-II Evolution

In [5]:
# Run evolution with DRY components
final_pop, stats = run_nsga2(
    data=data,
    config=config,
    create_individual_fn=create_random_individual,
    evaluate_fn=evaluate,
    crossover_fn=course_aware_crossover,
    mutate_fn=lambda ind: smart_mutation(ind, data),  # Closure over data
)

  Gen   0: min_hard=4523, min_soft= 2577, feasible=0/50
  Gen  20: min_hard=2008, min_soft= 1037, feasible=0/50
  Gen  40: min_hard=1722, min_soft=  797, feasible=0/50
  Gen  60: min_hard=1588, min_soft=  687, feasible=0/50
  Gen  80: min_hard=1522, min_soft=  604, feasible=0/50
  Gen  99: min_hard=1495, min_soft=  559, feasible=0/50
 Evolution complete in 88.2s


## 6. Results & Visualization

In [6]:
# Get best solution
best = get_best_individual(final_pop)
breakdown = get_constraint_breakdown(best, data)

# Print summary
print_summary(final_pop, stats, breakdown)

# Plot results
plot_convergence(stats, OUTPUT_DIR / "mode_a_convergence.png", title_prefix="Mode A: ")
plot_constraint_breakdown(breakdown, OUTPUT_DIR / "mode_a_breakdown.png", title="Mode A: Constraint Violations")


 EVOLUTION SUMMARY

 Best Solution:
   Hard Violations: 1495
   Soft Penalty:    696.0
   Feasible:         No

 Final Population (n=50):
   Feasible:     0/50 (0.0%)
   Min Hard:     1495
   Avg Hard:     1508.9
   Min Soft:     559.0
   Avg Soft:     672.9

️ Execution Time: 88.2s

 Best Solution Constraint Breakdown:
    course_completeness: 0
    instructor_exclusivity: 121
    instructor_qualifications: 19
    instructor_schedule_compactness: 77
    instructor_time_availability: 467
    paired_cohort_practical_alignment: 0
    room_exclusivity: 297
    room_suitability: 0
    room_time_availability: 0
    session_continuity: 75
    student_group_exclusivity: 591
    student_lunch_break: 306
    student_schedule_compactness: 238

   Total Hard: 1028, Total Soft: 1163.0

 Saved: ../output/mode_a_baseline/20260122_123618/mode_a_convergence.png


/home/krishna/Desktop/schedule-engine/src/schedule_engine/notebooks/viz.py:93: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 Saved: ../output/mode_a_baseline/20260122_123618/mode_a_breakdown.png


/home/krishna/Desktop/schedule-engine/src/schedule_engine/notebooks/viz.py:173: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Full Production Export (Optional)

Run this cell to generate the same outputs as CLI production runs:
- `schedule.json` - Full decoded schedule
- `calendar.pdf` - Visual calendar
- `plots/constraints/` - Constraint trend plots
- `plots/nsga/` - NSGA metrics plots
- `csv/` - Evolution data CSVs

In [7]:
# Reload export module with fixed inline decode
import importlib
import schedule_engine.notebooks.export
importlib.reload(schedule_engine.notebooks.export)

from schedule_engine.notebooks.export import export_full_results

# Generate all production outputs (same as CLI)
export_paths = export_full_results(
    population=final_pop,
    stats=stats,
    data=data,
    output_dir=OUTPUT_DIR,
    mode_name="mode_a_baseline",
)

# Show output location
print(f"\n All files saved to: {export_paths['output_dir']}")

 Saved: ../output/mode_a_baseline/20260122_123618/mode_a_baseline_schedule.json
 Saved: ../output/mode_a_baseline/20260122_123618/mode_a_baseline_stats.csv
 Saved: ../output/mode_a_baseline/20260122_123618/mode_a_baseline_summary.json

 All exports complete: ../output/mode_a_baseline/20260122_123618

 All files saved to: ../output/mode_a_baseline/20260122_123618
